# M4 · Eigenvectors, SVD, and Low Rank — companion notebook

> **Play with this.** A *demonstration, not an assessment* — the module's real assessment is its problem set. This notebook lets you run the rank slider's experiment with real numpy calls: truncate an image, watch the error curve, compare the spectra of structured versus random matrices (the empirical fact low-rank adaptation rests on), and recover planted latent structure from a synthetic survey with PCA.

Companion to the **Eigenvectors, SVD, and Low Rank** module of the Mathematical Foundations track at [llmsforsocialscience.net](https://llmsforsocialscience.net/).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(4)

## 1 · A matrix, taken apart

The same procedural "sun over hills" image the module's widget uses, decomposed by `np.linalg.svd`. The rank-1 expansion is a genuine equality — reconstruct with **all** terms and the error is machine precision.

In [ ]:
def make_image(size=48):
    y, x = np.mgrid[0:size, 0:size].astype(float)
    img = 0.85 - 0.3 * (y / size)                        # sky gradient
    sun = np.sqrt((x - 34) ** 2 + (y - 11) ** 2) < 6.5   # sun disc
    img[sun] = 0.98
    img[y > 30 + 4 * np.cos(x / 7.6)] = 0.38             # back hills
    img[y > 37 + 3 * np.cos(x / 4.2 + 1.3)] = 0.16       # front hills
    img += rng.normal(0, 0.02, img.shape)                # honest noise
    return np.clip(img, 0, 1)

A = make_image()
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("shapes:", U.shape, s.shape, Vt.shape)
print("full reconstruction error:", np.abs(U @ np.diag(s) @ Vt - A).max())

## 2 · The rank slider, by hand

Keep the first $r$ rank-1 pieces. Change `ranks` and re-run — at what $r$ do you first recognise the scene?

In [ ]:
def truncate(U, s, Vt, r):
    return U[:, :r] @ np.diag(s[:r]) @ Vt[:r, :]

ranks = [1, 2, 4, 8, 16, 48]
fig, axes = plt.subplots(1, len(ranks), figsize=(14, 2.6))
for ax, r in zip(axes, ranks):
    ax.imshow(truncate(U, s, Vt, r), cmap="gray", vmin=0, vmax=1)
    params = r * (48 + 48 + 1)
    ax.set_title(f"r={r}\n{params:,} params", fontsize=9)
    ax.axis("off")
plt.suptitle(f"full matrix: {48*48:,} params", y=1.08, fontsize=10)
plt.tight_layout(); plt.show()

## 3 · The error curve is the discarded spectrum

The module's identity $\lVert A - A_r \rVert_F = \sqrt{\sum_{i>r}\sigma_i^2}$, verified numerically at every rank at once.

In [ ]:
measured = [np.linalg.norm(A - truncate(U, s, Vt, r)) for r in range(len(s) + 1)]
predicted = [np.sqrt((s[r:] ** 2).sum()) for r in range(len(s) + 1)]
print("max |measured - predicted|:", np.abs(np.array(measured) - np.array(predicted)).max())

plt.figure(figsize=(7, 3.5))
plt.plot(measured, label="measured ‖A − A_r‖")
plt.plot(predicted, "--", label="√(discarded σ²)")
plt.xlabel("rank r kept"); plt.ylabel("Frobenius error"); plt.legend(); plt.tight_layout(); plt.show()

## 4 · Structured vs random: the fact LoRA rests on

A structured matrix concentrates its energy in a few singular values; a pure-noise matrix of the same size spreads it almost flat. Learned weight matrices in real models behave like the structured one — that empirical observation is the bedrock under low-rank methods. (numpy-only stand-in here: the module's reading, Hu et al. §4.1, shows the measurement on real transformer weights.)

In [ ]:
structured = make_image()
random_mat = rng.standard_normal((48, 48)) * structured.std() + structured.mean()

s_struct = np.linalg.svd(structured, compute_uv=False)
s_rand = np.linalg.svd(random_mat, compute_uv=False)

plt.figure(figsize=(7, 3.5))
plt.semilogy(s_struct, label="structured (image)")
plt.semilogy(s_rand, label="random noise, same size")
plt.xlabel("index i"); plt.ylabel("σ_i (log)"); plt.legend(); plt.tight_layout(); plt.show()

for name, sv in [("structured", s_struct), ("random", s_rand)]:
    frac = np.sqrt((sv[:8] ** 2).sum() / (sv ** 2).sum())
    print(f"{name:10s}: rank-8 keeps {100*frac:.1f}% of the Frobenius norm")

## 5 · PCA recovers planted structure

A synthetic 200-respondent, 30-question survey generated from **three** latent attitudes. PCA — the SVD of the centred data — finds the dimensionality without being told.

In [ ]:
n_resp, n_q, n_latent = 200, 30, 3
latent = rng.standard_normal((n_resp, n_latent))          # each respondent's 3 true attitudes
loadings = rng.standard_normal((n_latent, n_q))           # how questions tap attitudes
survey = latent @ loadings + rng.normal(0, 0.4, (n_resp, n_q))

Xc = survey - survey.mean(axis=0)                          # centre (M1) — load-bearing!
U2, s2, Vt2 = np.linalg.svd(Xc, full_matrices=False)
explained = s2 ** 2 / (s2 ** 2).sum()

plt.figure(figsize=(7, 3.2))
plt.bar(range(1, 16), explained[:15])
plt.xlabel("principal component"); plt.ylabel("explained variance share")
plt.title("Three planted attitudes, three dominant components")
plt.tight_layout(); plt.show()

print(f"top 3 components explain {100*explained[:3].sum():.1f}% of variance")
print(f"components 4-30 together: {100*explained[3:].sum():.1f}%  (noise floor)")

The thirty-column survey is, for practical purposes, a rank-3 matrix: it measures three things, asked thirty ways.

**The standing caution:** the components arrive unlabelled. PC1 is whatever *varies most* — for text embeddings that is often word frequency, not meaning (module problem 5 has the check and the fix).

## 6 · The LoRA arithmetic

The parameter table for a real transformer-sized matrix, computed rather than asserted.

In [ ]:
d = 4096
full = d * d
print(f"{'rank r':>8} {'params 2dr':>14} {'saving':>10}")
for r in [1, 4, 8, 16, 64, 256, 2048]:
    low = 2 * d * r
    print(f"{r:>8} {low:>14,} {full/low:>9.0f}×")
print(f"\nfull matrix: {full:,} parameters — break-even at r = d/2 = {d//2}")

Useful ranks in practice sit at 4–16 — three orders of magnitude below break-even. That gap is not algebra; it is an empirical statement about what fine-tuning *changes*, and it is why the method works.

---

**Next:** M6 · Optimisation — where the gradient from M5 finally gets used, and the learning rate becomes the one knob that matters.